# Preprocess MovieLens

Turns the raw CSVs into the arrays the model trains on. Change `DATASET` and re-run all cells to rebuild.

Outputs land in `data/processed/<dataset>/`:
- `train.parquet`, `val.parquet`, `test.parquet`: `user_idx`, `movie_idx`, `rating`, `timestamp`
- `movies.parquet`: one row per `movie_idx` (title, year, TMDB id, ...)
- `users.parquet`: `user_idx` to original `userId`
- `meta.json`: counts, thresholds and the training global mean

In [1]:
import json
from pathlib import Path

import pandas as pd

DATASET = "ml-latest-small"   # or "ml-32m"
MIN_MOVIE_RATINGS = 20
MIN_USER_RATINGS = 5

RAW = Path("../data/raw") / DATASET
OUT = Path("../data/processed") / DATASET

## 1. Load

In [2]:
raw = pd.read_csv(
    RAW / "ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"},
)
print(f"{len(raw):,} ratings, {raw['userId'].nunique():,} users, {raw['movieId'].nunique():,} movies")
raw.head()

100,836 ratings, 610 users, 9,724 movies


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## 2. Filter sparse movies and users

Removing movies can push a user below the threshold and vice versa, so this repeats until both conditions hold at once.

In [3]:
ratings = raw
while True:
    n_before = len(ratings)
    movie_counts = ratings.groupby("movieId")["rating"].transform("size")
    ratings = ratings[movie_counts >= MIN_MOVIE_RATINGS]
    user_counts = ratings.groupby("userId")["rating"].transform("size")
    ratings = ratings[user_counts >= MIN_USER_RATINGS]
    if len(ratings) == n_before:
        break

print(f"kept {len(ratings):,} of {len(raw):,} ratings ({len(ratings) / len(raw):.1%})")
print(f"kept {ratings['movieId'].nunique():,} of {raw['movieId'].nunique():,} movies")
print(f"kept {ratings['userId'].nunique():,} of {raw['userId'].nunique():,} users")

kept 67,898 of 100,836 ratings (67.3%)
kept 1,297 of 9,724 movies
kept 610 of 610 users


## 3. Remap ids to contiguous indices

Embedding tables need indices 0..N-1; MovieLens ids have gaps.

In [4]:
users = pd.DataFrame({"userId": sorted(ratings["userId"].unique())})
users.insert(0, "user_idx", range(len(users)))
movie_index = pd.DataFrame({"movieId": sorted(ratings["movieId"].unique())})
movie_index.insert(0, "movie_idx", range(len(movie_index)))

ratings = ratings.merge(users, on="userId").merge(movie_index, on="movieId")
ratings = ratings[["user_idx", "movie_idx", "rating", "timestamp"]].astype(
    {"user_idx": "int32", "movie_idx": "int32"}
)
ratings.head()

,user_idx,movie_idx,rating,timestamp
0,0,0,4.0,964982703
1,0,2,4.0,964981247
2,0,4,4.0,964982224
3,0,24,5.0,964983815
4,0,26,5.0,964982931


## 4. Split by time

Each user's most recent rating goes to test, the one before it to validation, the rest to train.

In [5]:
ratings = ratings.sort_values(["user_idx", "timestamp", "movie_idx"], kind="stable")
from_end = ratings.groupby("user_idx").cumcount(ascending=False)
train = ratings[from_end >= 2]
val = ratings[from_end == 1]
test = ratings[from_end == 0]

# A held-out movie the model never saw in training has an untrained vector; don't score on it.
seen = set(train["movie_idx"].unique())
val = val[val["movie_idx"].isin(seen)]
test = test[test["movie_idx"].isin(seen)]

train, val, test = (df.reset_index(drop=True) for df in (train, val, test))
print(f"train {len(train):,}   val {len(val):,}   test {len(test):,}")

train 66,678   val 610   test 610


## 5. Movie table

Splits `Usual Suspects, The (1995)` into name `The Usual Suspects` and year `1995`, and attaches TMDB ids. Trailing parentheticals are alternate titles and are dropped from `name`; the original stays in `title`.

In [6]:
# Articles MovieLens moves to the end of a title.
ARTICLES = "The|A|An|Le|La|Les|L'|Il|Lo|I|Gli|El|Los|Las|Un|Une|Una|Der|Die|Das|Den|Det|De|O|Os"

titles = pd.read_csv(RAW / "movies.csv", dtype={"movieId": "int32"})
links = pd.read_csv(RAW / "links.csv", dtype={"movieId": "int32", "imdbId": "string", "tmdbId": "Int64"})
movies = movie_index.merge(titles, on="movieId", how="left").merge(links, on="movieId", how="left")

title = movies["title"].str.strip()
movies["year"] = title.str.extract(r"\((\d{4})\)\)?$")[0].astype("Int64")
name = title.str.replace(r"\s*\(\d{4}\)\)?$", "", regex=True)
name = name.str.replace(r"(?<=\S)(\s*\([^()]*\))+$", "", regex=True)
movies["name"] = name.str.replace(rf"^(.*), ({ARTICLES})$", r"\2 \1", regex=True)

n_ratings = train.groupby("movie_idx").size().rename("n_train_ratings")
movies = movies.merge(n_ratings, on="movie_idx", how="left")
movies = movies[
    ["movie_idx", "movieId", "title", "name", "year", "genres", "imdbId", "tmdbId", "n_train_ratings"]
]
movies.sample(10, random_state=0)

,movie_idx,movieId,title,name,year,genres,imdbId,tmdbId,n_train_ratings
665,665,2915,Risky Business (1983),Risky Business,1983,Comedy,0086200,9346,41
474,474,1911,Dr. Dolittle (1998),Dr. Dolittle,1998,Comedy,0118998,3050,32
717,717,3254,Wayne's World 2 (1993),Wayne's World 2,1993,Comedy,0108525,8873,27
240,240,910,Some Like It Hot (1959),Some Like It Hot,1959,Comedy|Crime,0053291,239,49
699,699,3101,Fatal Attraction (1987),Fatal Attraction,1987,Drama|Thriller,0093010,10998,36
312,312,1203,12 Angry Men (1957),12 Angry Men,1957,Drama,0050083,389,54
1195,1195,80549,Easy A (2010),Easy A,2010,Comedy|Romance,1282140,37735,27
575,575,2369,Desperately Seeking Susan (1985),Desperately Seeking Susan,1985,Comedy|Drama|Romance,0089017,8130,24
1018,1018,8949,Sideways (2004),Sideways,2004,Comedy|Drama|Romance,0375063,9675,39
211,211,748,"Arrival, The (1996)",The Arrival,1996,Action|Sci-Fi|Thriller,0115571,10547,21


## 6. Save

In [7]:
meta = {
    "dataset": DATASET,
    "min_movie_ratings": MIN_MOVIE_RATINGS,
    "min_user_ratings": MIN_USER_RATINGS,
    "n_users": len(users),
    "n_movies": len(movies),
    "n_raw_ratings": len(raw),
    "n_train": len(train),
    "n_val": len(val),
    "n_test": len(test),
    "global_mean": float(train["rating"].mean()),
}

OUT.mkdir(parents=True, exist_ok=True)
train.to_parquet(OUT / "train.parquet", index=False)
val.to_parquet(OUT / "val.parquet", index=False)
test.to_parquet(OUT / "test.parquet", index=False)
movies.to_parquet(OUT / "movies.parquet", index=False)
users.to_parquet(OUT / "users.parquet", index=False)
(OUT / "meta.json").write_text(json.dumps(meta, indent=2) + "\n")
meta

{'dataset': 'ml-latest-small',
 'min_movie_ratings': 20,
 'min_user_ratings': 5,
 'n_users': 610,
 'n_movies': 1297,
 'n_raw_ratings': 100836,
 'n_train': 66678,
 'n_val': 610,
 'n_test': 610,
 'global_mean': 3.6213443279266357}

## 7. Checks

In [8]:
everything = pd.concat([train, val, test])
assert train["user_idx"].nunique() == meta["n_users"] and train["user_idx"].max() == meta["n_users"] - 1
assert train["movie_idx"].nunique() == meta["n_movies"] and train["movie_idx"].max() == meta["n_movies"] - 1
assert not everything.duplicated(["user_idx", "movie_idx"]).any()
assert set(val["movie_idx"]) <= seen and set(test["movie_idx"]) <= seen

last_train = train.groupby("user_idx")["timestamp"].max()
assert (val.set_index("user_idx")["timestamp"] >= last_train.reindex(val["user_idx"]).values).all()
assert (test.set_index("user_idx")["timestamp"] >= last_train.reindex(test["user_idx"]).values).all()
print("all checks passed")

all checks passed
